Visualise model output from CF registry data vs baseline ppFEV1

In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import cfr.cfr_viz_helpers as vh

In [ ]:
# Load AC with inferred from 2023 data, 2nd day = 2019 data

df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)

# EXCEL
# 2 entries means there is a 2019 entry for every 2023 entry
# df_res = bd.load_meas_from_excel(
#     "infer_AR_using_19_23_data_2entries_fev1_10122025",
#     # "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
#     study_folder="CFR",
#     str_cols_to_arrays=["Airway resistance (%)"],
# )
# df_res = df_res.drop(columns=["Healthy FEV1 (L)"])

# CSV
# df_res = (
#     bd.load_meas_from_excel(
#         # "infer_AR_using_two_days_model_19_23_data_2entries_fev1_10122025",
#         "infer_AR_using_two_days_model_19_23_data_2entries_fev1_fef2575_10122025",
#         study_folder="CFR",
#         str_cols_to_arrays=["Airway resistance (%)"],
#         use_csv=True,
#         date_cols=["Day"],
#         bypass_sanity_checks=True,
#     )
#     .drop(columns=["Healthy FEV1 (L)"])
#     .rename(columns={"Day": "Date Recorded"})
# )

# Merging
df = df_res.merge(df_meas, on=["ID", "Date Recorded"])

In [2]:
# Load AC from 2019 data with 2nd day = best FEV1 (no FEF2575)
# df = bd.load_meas_from_excel("AR_19_data_with_best_FEV1", study_folder="CFR", str_cols_to_arrays=["Airway resistance (%)"])
df = bd.load_meas_from_excel(
    "infer_all_19_data_with_best_FEV1",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(FEV1|HFEV1_pers)",
        "P(HFEV1|FEF2575, bFEV1)",
    ],
)
print(f"Shape: {df.shape}")

Shape: (2037, 21)


In [4]:
# Process

# Keep only values from 2023
# df23 = df[df["Date Recorded"] == datetime.date(2023, 1, 1)]

df23 = df

AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df23[AC.name] = df23[AR.name].apply(lambda arr: arr[::-1])
df23["ecFEV1 % Predicted (clipped)"] = df23["ecFEV1 % Predicted"].clip(upper=100)
df23["P(ppFEV1|AC)"] = vh.calc_P_ppFEV1_given_AC(df23, AC)
df["P(ppFEV1|AC) ratioed"] = vh.calc_P_ppFEV1_given_AC(df, AC, corr=True)

# Airway conductance

In [36]:
## FILL ##
ratioed = False
prctile = 10

df_to_plot, t = vh.filter_confidently_disagreeing_examples(df, prctile, ratioed)
# df_to_plot = df23[df23["P(ppFEV1|AC)"] <= t]

# title = f"Dumbell plot for CF Registry 2023, 2019 2nd day, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, 2019 2nd day, FEV1 & FEF25-75 (2entries), {t*100:.2f}% conf. disagreeing"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1 (no FEF25-75)"
title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 & FEF2575 (2entries)"

ac_col = AC.name

fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

ppfev1_row = "ecFEV1 % Predicted (clipped)"
# ppfev1_row = "ecFEV1 % Predicted"

df_to_plot, _, _ = vh.get_dumbell_plot_data(
    df_to_plot, AC.name, AC, ppfev1_row=ppfev1_row
)

# Split dataframe between mild, moderate and severe CF lung disease
# Equivalent to Mean AR_ecFEV1% < 30%, 30 to 60 and > 60%
# Get unique IDs and their corresponding Mean AR_ecFEV1% values
mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
# mask_ppfev1 = df_to_plot["measure"] == f"Mean {ac_col} prediction"

id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)}) clipped"

# Print the sizes to verify
print(f"# Mild: {len(mild_ids)}")
print(f"# Moderate: {len(moderate_ids)}")
print(f"# Severe: {len(severe_ids)}")

# Plot the three groups
vh.plot_dumbell_for_df(
    fig, df_mild, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 3
)
vh.plot_dumbell_for_df(
    fig, df_moderate, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 2
)
vh.plot_dumbell_for_df(
    fig, df_severe, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 1
)


fig.update_layout(
    height=1000,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="Airway conductance (%)",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(
    # f"{dh.get_path_to_main()}PlotsBreathe/Dumbell_plot_AR_ecFEV1_by_severity/{title}_clipped.pdf"
    # f"{dh.get_path_to_main()}PlotsCFR/{title}_clipped.pdf"
    f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf"
)
# fig.show()

# On this plot, if a person is very sick, it's best fev1 measurement will contain a lot of inflammatory markers (sputum, airway wall inflammation).
# The sicker the person,the more underestimated the AR pred is because the maximum FEV1 blown is not healthy.
# Let's add the FEF25-75.

# 60% of data falls within red band

# TODO: add vertical lines corresponding to key AR values

# Longitudinal AR profile on the web app

# Mild: 169
# Moderate: 33
# Severe: 2
35.60473898878335


# FEV1 % personalised predicted (Non saturating)

In [9]:
ecFEV1 = mh.VariableNode("ecFEV1 (L)", 0, 6, 0.05, prior=None)
df["mean P(FEV1|HFEV1_pers)"] = df.apply(
    lambda row: ecFEV1.get_mean(row["P(FEV1|HFEV1_pers)"]), axis=1
)

df["FEV1%PersPred"] = df["FEV1"] / df["mean P(FEV1|HFEV1_pers)"] * 100

In [16]:
ecFEV1.get_mean(df.loc[1,'P(FEV1|HFEV1_pers)'])

1.3465106081635057

In [15]:
df.loc[1,'FEV1']

2.670000076293945

In [10]:
df.head()

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,idx FEV1,idx FEF2575%FEV1,idx best FEV1,"P(HFEV1|FEF2575, bFEV1)",P(FEV1|HFEV1_pers),P(FEV1_obs|FEV1),Airway resistance (%),mean P(FEV1|HFEV1_pers),FEV1%PersPred,ppFEV1 - pppFEV1
0,B155916,32,162,1.50,0.47,NaN,Female,2019-01-01,1.50,0.47,...,30,15,0,"[0.0108908472, 0.0168525629, 0.0255577133, 0.0...","[5.03305522e-195, 6.92721681e-172, 1.35042021e...",2.734816e-02,"[0.000696205794, 0.00226151902, 0.00857707029,...",1.494502,100.367888,-52.883650
1,B155917,45,175,2.67,1.00,NaN,Male,2019-01-01,2.67,1.00,...,53,18,0,"[0.0389828207, 0.0497619982, 0.0626672801, 0.0...","[1.80153743e-194, 2.47953575e-171, 4.83370924e...",1.274619e-07,"[0.0256364772, 0.0835114084, 0.232441544, 0.27...",1.346511,198.290311,-130.476402
2,B155918,34,191,4.82,3.48,NaN,Male,2019-01-01,4.82,3.48,...,96,36,0,"[0.0529837476, 0.0639540381, 0.0762691971, 0.0...","[2.448571e-194, 3.37007672e-171, 6.56976652e-1...",7.058962e-34,"[0.675293396, 0.279674216, 0.0438870872, 0.001...",1.310168,367.891790,-274.149501
3,B155921,34,150,1.44,0.59,NaN,Female,2019-01-01,1.44,0.59,...,28,20,0,"[0.0082905348, 0.0136774835, 0.0219876514, 0.0...","[3.83135661e-195, 5.27326578e-172, 1.02799218e...",9.708870e-02,"[0.00386665096, 0.00871641119, 0.0161529033, 0...",1.492071,96.510152,-42.248535
4,B155925,38,167,0.92,0.34,NaN,Female,2019-01-01,0.92,0.34,...,18,18,0,"[0.0137607135, 0.0206453576, 0.0303857943, 0.0...","[6.35932442e-195, 8.75261986e-172, 1.70627181e...",1.016236e-03,"[2.96740131e-06, 2.291855e-05, 0.000116745726,...",1.468220,62.660897,-34.303799


In [4]:
import plotly.graph_objects as go

In [5]:
def plot_dumbell_for_df_model_ppfev1(fig, df, measures, col):
    ac_mean = measures[0]
    baseline = measures[1]

    mask = df["measure"] == ac_mean
    fig.add_trace(
        go.Scatter(
            x=df[mask]["value"],
            y=df[mask]["ID"],
            mode="markers",
            marker=dict(color="red", size=4),
            name="ecFEV1 % pers. pred."
        ),
        row=1,
        col=col,
    )

    mask = df["measure"] == baseline
    ecfev1_prct_pred = df[mask]["value"]
    # Where above 100, set to 100
    # ecfev1_prct_pred = np.clip(ecfev1_prct_pred, 0, 100)
    fig.add_trace(
        go.Scatter(
            x=ecfev1_prct_pred,
            y=df[mask]["ID"],
            mode="markers",
            name="ecFEV1 % predicted",
            marker=dict(size=4, color="blue"),
        ),
        row=1,
        col=col,
    )

In [6]:
def get_dumbell_plot_data_model_ppfev1(df, ac_row, ppfev1_row="ecFEV1 % Predicted"):
    # Avoid modifying the original dataframe
    df_res = df.copy()

    ids_sorted = df_res.sort_values("ppFEV1 - pppFEV1", ascending=False)["ID"].values

    df_melted = (
        df_res.melt(
            id_vars=["ID"],
            value_vars=[ppfev1_row, ac_row],
            var_name="measure",
            value_name="value",
        )
        .set_index("ID")
        .loc[ids_sorted]
        .reset_index()
    )

    return df_melted, df_res, ids_sorted

In [8]:
## FILL ##
prctile = 0

title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, FEV1%PersPred {prctile:.0f}th prctile"
ac_col = "FEV1%PersPred"
ppfev1_row = "ecFEV1 % Predicted"

df["ppFEV1 - pppFEV1"] = df["FEV1 % Predicted"] - df["FEV1%PersPred"]
col = "ppFEV1 - pppFEV1"
t = df[col].abs().quantile(prctile / 100)
df_to_plot = df[df[col].abs() > t]


df_to_plot, _, _ = get_dumbell_plot_data_model_ppfev1(
    df_to_plot, ac_col, ppfev1_row
)

mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

# Plot the three groups
fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)
title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)})"
plot_dumbell_for_df_model_ppfev1(fig, df_mild, [ac_col, ppfev1_row], 3)
plot_dumbell_for_df_model_ppfev1(fig, df_moderate, [ac_col, ppfev1_row], 2)
plot_dumbell_for_df_model_ppfev1(fig, df_severe, [ac_col, ppfev1_row], 1)

fig.update_layout(
    height=1000,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="FEV1 % predicted",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
# fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
fig.show()

13.3130788065074
